In [ ]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import LineString
from shapely.ops import unary_union
import matplotlib.pyplot as plt

os.getcwd()

In [ ]:
fault_type = 'crustal'
fault_name = None  # None for all faults, or e.g. 'NW Cardrona North'

min_slip = 0.0
max_slip = 50

footbuff = 5000
hangingbuff = 5000   # 15000
edgebuff = 3000   # 3000

if fault_type == 'crustal' or fault_type.upper() == 'CFM':
    datadir = "./crustal/discretised_CFM"
    fault_type = 'CFM'
    polygon_file = "named_rectangle_polygons.geojson"
    trace_file = "name_filtered_fault_sections.geojson"
    traces = gpd.read_file(f"{datadir}/{trace_file}").to_crs(2193)
    if fault_name:
        outname = f"{fault_name.replace(' ', '-')}_hang-{hangingbuff / 1000:.0f}km_foot-{footbuff / 1000:.0f}km_edge-{edgebuff / 1000:.0f}km"
        traces = traces[traces['ParentName'] == fault_name]
    else:
        # traces = traces[traces['SlipRate'] > min_slip]
        # traces = traces[traces['SlipRate'] < max_slip]
        outname = f"{fault_type}_hang-{hangingbuff / 1000:.0f}km_foot-{footbuff / 1000:.0f}km_edge-{edgebuff / 1000:.0f}km"
else:
    if fault_type == 'py' or fault_type == 'puysegur':
        datadir = "./subduction/discretised_puysegur"
        fault_type = 'py'
    else:
        datadir = "./subduction/discretised_fq_hikkerm"
        fault_type= "sz"
    
    outname = f"{fault_type}_hang-{hangingbuff / 1000:.0f}km_foot-{footbuff / 1000:.0f}km_edge-{edgebuff / 1000:.0f}km"
    polygon_file = f"{fault_type}_all_rectangle_outlines.geojson"
    trace_file = f"{fault_type}_all_filtered_fault_sections.geojson"

polygons = gpd.read_file(f"{datadir}/{polygon_file}").to_crs(2193)

if fault_type == 'CFM':
    polygons = polygons.loc[traces.index]
else:
    traces = pd.DataFrame({'FaultID': polygons['fault_id'], 'ParentName': [f"{fault_type}_{a}" for a in range(polygons.shape[0])], 'SlipRate': [0] * polygons.shape[0], 'DipDir': [180] * polygons.shape[0], 'DipDeg': [0] * polygons.shape[0]})
    # traces = pd.DataFrame({'FaultID': polygons['fault_id'], 'ParentName': [0] * polygons.shape[0], 'SlipRate': [0] * polygons.shape[0], 'DipDir': [180] * polygons.shape[0], 'DipDeg': [0] * polygons.shape[0]})

In [ ]:
buff_poly = gpd.GeoDataFrame([], columns=['FaultName', 'SlipRate', 'DipDir', 'geometry'], geometry='geometry', crs=2193)
old_poly = gpd.GeoDataFrame([], columns=['FaultName', 'SlipRate', 'DipDir', 'geometry'], geometry='geometry', crs=2193)

for name, group in traces.groupby('ParentName'):
    # if name.lower() != 'te anau'.lower():
    #     continue
    print(name)
    poly_buffs = []
    old_polys = []
    for ix, (_, row) in enumerate(group.iterrows()):
        if ix == 0:
            slipRate = row['SlipRate']
            dipDir = row['DipDir']
            factor = 1
        
        coords = polygons.loc[row['FaultID']].geometry.boundary.coords
        factor = -1 if coords[0][1] > coords[1][1] else 1
        if row['DipDir'] > 180:
            factor *= -1
        # if ix == 0 or 'Taramakau' in name:
        #     plt.plot(coords[0][0], coords[0][1], 'r.')
        #     plt.plot(coords[1][0], coords[1][1], 'g.')
        #     plt.plot(coords[2][0], coords[2][1], 'b.')
        #     plt.plot(coords[3][0], coords[3][1], 'k.')
        #     if 'Taramakau' in name:
        #         print(row)
        #         plt.title(f"{polygons.loc[row['FaultID']]['fault_name']}: {row['DipDir']}")
        #     else:
        #         plt.title(f"{name}: {row['DipDir']}")
        #     plt.show()
        if 'sz' in fault_type:
            coords = [coords[1], coords[2], coords[3], coords[0]]
        top = LineString([coords[0], coords[1]]).buffer(footbuff * factor, single_sided=True)
        right = LineString([coords[1], coords[2]]).buffer(edgebuff * factor, single_sided=True)
        base = LineString([coords[2], coords[3]]).buffer([hangingbuff if row['DipDeg'] != 90 else footbuff][0] * factor, single_sided=True)
        left = LineString([coords[3], coords[0]]).buffer(edgebuff * factor, single_sided=True)
        poly_buffs = poly_buffs + [top, right, base, left]
        old_polys = old_polys + [polygons.loc[row['FaultID']].geometry]
    old_union = unary_union(old_polys)
    conv_hull = unary_union(poly_buffs).convex_hull

    new_fault = gpd.GeoDataFrame({"FaultName": [name], 'SlipRate': [slipRate], 'DipDir': [dipDir], "geometry": [conv_hull]}, crs=buff_poly.crs)
    old_fault = gpd.GeoDataFrame({"FaultName": [name], 'SlipRate': [slipRate], 'DipDir': [dipDir], "geometry": [old_union]}, crs=buff_poly.crs)
    buff_poly = pd.concat([buff_poly, new_fault], ignore_index=True)
    old_poly = pd.concat([old_poly, old_fault], ignore_index=True)


os.makedirs('./sites/poly_buffers', exist_ok=True)
buff_poly.to_file(f'./sites/poly_buffers/{outname}.geojson', driver='GeoJSON')
old_poly.to_file(f'./sites/poly_buffers/{fault_type}_faultsections.geojson', driver='GeoJSON')
print(f'./sites/poly_buffers/{outname}.geojson')